In [1]:
from utils import *
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
# device = "cpu"
print(f"Using {device} device")

Using mps device


## Linear Regression

Simple population-level linear regression model. The country is not considered.

In [3]:
X, Y, C = load_height_data()
model = LinearRegression().fit(X, Y)
y_pred = model.predict(X)
print("Loss:\n", np.mean((y_pred - Y)**2))
print("RMSD:\n", np.sqrt(np.mean((y_pred - Y)**2)))

Loss:
 45.53098
RMSD:
 6.747665


Next, we try learning a separate linear regression model for each country. Notice the loss is much better.

In [4]:
X, Y, C = load_height_data()
model = ContextLinearRegression().fit(X, Y, C)
y_pred = model.predict(X, C)
print("Loss:\n", np.mean((y_pred - Y)**2))
print("RMSD:\n", np.sqrt(np.mean((y_pred - Y)**2)))

Loss:
 23.753902446756058
RMSD:
 4.87379753854795


## Multilayer Perceptron
The MLP without country as input performs about equally well as learning a separate linear regression model for each country.

In [8]:
model = NeuralNetwork(dim_in=3, dim_out=1, dim_hidden=300, n_hidden=1).to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

full_dataset = HeightDataset()

training_data, test_data = torch.utils.data.random_split(full_dataset, [0.8, 0.2])
train_dataloader = DataLoader(training_data, batch_size=50)
test_dataloader = DataLoader(test_data, batch_size=50)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer, device)
    test_loss = test(test_dataloader, model, loss_fn, device)
print("Done!")

print("Loss:", test_loss.item())
print("RMSD:", np.sqrt(test_loss.item()))

Epoch 1
-------------------------------
loss: 369.501465  [   50/168000]
loss: 31.514580  [ 5050/168000]
loss: 47.903862  [10050/168000]
loss: 39.276512  [15050/168000]
loss: 42.566490  [20050/168000]
loss: 21.084980  [25050/168000]
loss: 29.106981  [30050/168000]
loss: 22.486507  [35050/168000]
loss: 30.810459  [40050/168000]
loss: 31.797680  [45050/168000]
loss: 27.958963  [50050/168000]
loss: 29.816790  [55050/168000]
loss: 25.977684  [60050/168000]
loss: 29.277449  [65050/168000]
loss: 21.942265  [70050/168000]
loss: 26.116110  [75050/168000]
loss: 23.394178  [80050/168000]
loss: 30.113607  [85050/168000]
loss: 24.991022  [90050/168000]
loss: 25.699638  [95050/168000]
loss: 21.786543  [100050/168000]
loss: 25.972841  [105050/168000]
loss: 21.396269  [110050/168000]
loss: 20.809658  [115050/168000]
loss: 31.596205  [120050/168000]
loss: 32.794106  [125050/168000]
loss: 27.826141  [130050/168000]
loss: 16.638855  [135050/168000]
loss: 21.055878  [140050/168000]
loss: 25.392729  [1450

## Context-Sensitive Network
The task is one-hot-encoded and appended to the input.

In [4]:
model = NeuralNetwork(dim_in=203, dim_out=1, dim_hidden=300, n_hidden=3).to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

full_dataset = ContextSensitiveHeightDataset()

training_data, test_data = torch.utils.data.random_split(full_dataset, [0.8, 0.2])
train_dataloader = DataLoader(training_data, batch_size=50)
test_dataloader = DataLoader(test_data, batch_size=50)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer, device)
    test_loss = test(test_dataloader, model, loss_fn, device)
print("Done!")

print("Loss:", test_loss.item())
print("RMSD:", np.sqrt(test_loss.item()))

Epoch 1
-------------------------------
loss: 415.010864  [   50/168000]
loss: 35.103355  [ 5050/168000]
loss: 30.059906  [10050/168000]
loss: 46.207333  [15050/168000]
loss: 45.149559  [20050/168000]
loss: 26.662819  [25050/168000]
loss: 23.492069  [30050/168000]
loss: 39.285503  [35050/168000]
loss: 26.000284  [40050/168000]
loss: 22.232656  [45050/168000]
loss: 20.661480  [50050/168000]
loss: 27.616695  [55050/168000]
loss: 22.876326  [60050/168000]
loss: 25.211098  [65050/168000]
loss: 19.422127  [70050/168000]
loss: 20.598198  [75050/168000]
loss: 20.859013  [80050/168000]
loss: 19.862328  [85050/168000]
loss: 16.274414  [90050/168000]
loss: 23.849020  [95050/168000]
loss: 22.325928  [100050/168000]
loss: 18.108906  [105050/168000]
loss: 26.383043  [110050/168000]
loss: 22.909891  [115050/168000]
loss: 16.951044  [120050/168000]
loss: 19.458941  [125050/168000]
loss: 21.654589  [130050/168000]
loss: 16.674910  [135050/168000]
loss: 13.255246  [140050/168000]
loss: 17.267719  [1450

## Learned-Context Neural Network
Instead of one-hot encoding, the task is represented as a learned context parameter and appended to the input.

In [3]:
model = LearnedContextNN(dim_in=3, dim_out=1, dim_hidden=300, n_hidden=3, dim_context=30, n_context=200).to(device)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

full_dataset = ContextSensitiveHeightDataset()

training_data, test_data = torch.utils.data.random_split(full_dataset, [0.8, 0.2])
train_dataloader = DataLoader(training_data, batch_size=50)
test_dataloader = DataLoader(test_data, batch_size=50)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer, device)
    test_loss = test(test_dataloader, model, loss_fn, device)
print("Done!")

print("Loss:", test_loss.item())
print("RMSD:", np.sqrt(test_loss.item()))

Epoch 1
-------------------------------
loss: 432.886383  [   50/168000]
loss: 36.837124  [ 5050/168000]
loss: 29.185120  [10050/168000]
loss: 23.446230  [15050/168000]
loss: 29.418392  [20050/168000]
loss: 28.706867  [25050/168000]
loss: 37.618843  [30050/168000]
loss: 19.160088  [35050/168000]
loss: 17.414637  [40050/168000]
loss: 15.764810  [45050/168000]
loss: 17.302893  [50050/168000]
loss: 15.484543  [55050/168000]
loss: 10.799051  [60050/168000]
loss: 4.999252  [65050/168000]
loss: 11.890845  [70050/168000]
loss: 6.588329  [75050/168000]
loss: 9.466811  [80050/168000]
loss: 7.656347  [85050/168000]
loss: 5.336452  [90050/168000]
loss: 2.627325  [95050/168000]
loss: 4.612514  [100050/168000]
loss: 5.087598  [105050/168000]
loss: 8.295614  [110050/168000]
loss: 4.051041  [115050/168000]
loss: 4.038744  [120050/168000]
loss: 3.378821  [125050/168000]
loss: 2.255579  [130050/168000]
loss: 2.961540  [135050/168000]
loss: 1.631577  [140050/168000]
loss: 2.453357  [145050/168000]
loss: